# 🧪 Tutorial 6: Calibration Verification — SOLUTIONS

**FOR INSTRUCTOR USE ONLY**

In [ ]:
# Setup
options(repos = c(CRAN = "https://cloud.r-project.org"))
if (!requireNamespace("ggplot2", quietly = TRUE)) install.packages("ggplot2")
library(ggplot2)

In [ ]:
# Create the linearity data
linearity_data <- data.frame(
  Concentration = rep(c(0, 25, 50, 100, 200, 400), each = 3),
  Signal = c(
    0.052, 0.048, 0.055,
    0.245, 0.251, 0.238,
    0.478, 0.485, 0.472,
    0.952, 0.965, 0.948,
    1.875, 1.892, 1.868,
    3.425, 3.458, 3.412
  )
)

## Step 1: Load and Explore Data

In [ ]:
# SOLUTION: Check structure
str(linearity_data)
dim(linearity_data)  # 18 rows, 2 columns

In [ ]:
# SOLUTION: Summary by concentration
aggregate(Signal ~ Concentration, data = linearity_data, FUN = mean)

## Step 2: Visualize Raw Data

In [ ]:
# SOLUTION: Scatter plot
plot(linearity_data$Concentration, linearity_data$Signal,
     main = "Linearity Verification: Raw Data",
     xlab = "Concentration (mg/dL)",
     ylab = "Signal (OD)",
     pch = 19, col = "steelblue", cex = 1.5)

## Step 3: Fit Linear Model

In [ ]:
# SOLUTION: Fit model
model <- lm(Signal ~ Concentration, data = linearity_data)
summary(model)

In [ ]:
# SOLUTION: Extract parameters
intercept <- coef(model)[1]
slope <- coef(model)[2]
r_squared <- summary(model)$r.squared

cat(sprintf("Signal = %.6f × Concentration + %.4f\n", slope, intercept))
cat(sprintf("R² = %.4f\n", r_squared))

## Step 4: Create Calibration Plot

In [ ]:
# SOLUTION: Base R plot
plot(linearity_data$Concentration, linearity_data$Signal,
     main = "Calibration Curve",
     xlab = "Concentration (mg/dL)",
     ylab = "Signal (OD)",
     pch = 19, col = "steelblue", cex = 1.5)
abline(model, col = "red", lwd = 2)
legend("topleft",
       legend = c(sprintf("y = %.5fx + %.4f", slope, intercept),
                  sprintf("R² = %.4f", r_squared)),
       bty = "n")

In [ ]:
# SOLUTION: ggplot2 version
ggplot(linearity_data, aes(x = Concentration, y = Signal)) +
  geom_point(size = 3, color = "steelblue") +
  geom_smooth(method = "lm", se = TRUE, color = "red", fill = "pink", alpha = 0.3) +
  labs(
    title = "Calibration Curve: Linearity Verification",
    subtitle = sprintf("y = %.5fx + %.4f, R² = %.4f", slope, intercept, r_squared),
    x = "Concentration (mg/dL)",
    y = "Signal (OD)"
  ) +
  theme_minimal()

## Step 5: Residual Analysis

In [ ]:
# SOLUTION: Add residuals
linearity_data$Predicted <- predict(model)
linearity_data$Residual <- residuals(model)
print(linearity_data)

In [ ]:
# SOLUTION: Residual plot
plot(linearity_data$Concentration, linearity_data$Residual,
     main = "Residual Plot",
     xlab = "Concentration (mg/dL)",
     ylab = "Residual",
     pch = 19, col = "steelblue", cex = 1.5)
abline(h = 0, col = "red", lty = 2, lwd = 2)

In [ ]:
# SOLUTION: Diagnostic plots
par(mfrow = c(2, 2))
plot(model)
par(mfrow = c(1, 1))

## Step 6: Calculate Unknown Concentrations

In [ ]:
# SOLUTION: Calculate unknowns
unknown_samples <- data.frame(
  SampleID = c("Patient_A", "Patient_B", "Patient_C"),
  Signal = c(0.612, 1.534, 2.856)
)

unknown_samples$Concentration <- (unknown_samples$Signal - intercept) / slope
print(unknown_samples)

## Verification Report

In [ ]:
# SOLUTION: Complete report
cat("=== LINEARITY VERIFICATION REPORT ===\n\n")
cat("Date: January 21, 2026\n")
cat("Analyst: MLT Student\n")
cat("Instrument: Chemistry Analyzer\n\n")

cat("--- Calibration Parameters ---\n")
cat(sprintf("Slope: %.6f\n", slope))
cat(sprintf("Intercept: %.4f\n", intercept))
cat(sprintf("R²: %.4f\n\n", r_squared))

cat("--- Acceptance Criteria ---\n")
r2_threshold <- 0.99

if (r_squared >= r2_threshold) {
  cat(sprintf("R² Criterion: PASS (%.4f ≥ %.2f)\n", r_squared, r2_threshold))
} else {
  cat(sprintf("R² Criterion: FAIL (%.4f < %.2f)\n", r_squared, r2_threshold))
}

cat("\n--- Conclusion ---\n")
if (r_squared >= r2_threshold) {
  cat("Linearity verification: PASS\n")
  cat("The assay demonstrates acceptable linearity across the tested range.\n")
} else {
  cat("Linearity verification: FAIL\n")
  cat("Further investigation required.\n")
}